# Parts C and D — Support Copilot

This notebook demonstrates privacy-safe ticket triage, grounded policy retrieval, and bounded database-backed refund tools. The integrated FastAPI backend and React interface live under `backend/` and `frontend/`. Everything runs in deterministic mock mode without credentials or hosted-model network calls.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.ticket_triage import compare_prompt_strategies, run_triage
from src.policy_rag import run_policy_rag
from src.refund_agent import evaluate_refund_agent


## 1. Privacy-safe ticket triage

Empty and overlong text is blocked before model calls. Email, phone, and order identifiers are redacted. Transient failures are retried, output categories are allow-listed, and unresolved cases route to human review. Equivalent zero-shot and few-shot prompts are evaluated side by side.

In [2]:
prompt_comparison, prompt_summaries = compare_prompt_strategies()
display(prompt_comparison)
triage, triage_summary = run_triage(prompt_mode="few_shot")
print("Few-shot summary:", triage_summary)
display(triage.groupby(["status", "route"]).size().rename("tickets").reset_index())
display(triage.head())

,prompt_mode,classified,accuracy_on_classified,human_review,tokens_used,retries
0,zero_shot,147,0.8571,17,8469,19
1,few_shot,147,1.0000,17,13019,19


Few-shot summary: {'tickets': 164, 'prompt_mode': 'few_shot', 'classified': 147, 'accuracy_on_classified': 1.0, 'token_budget': 100000, 'tokens_used': 13019, 'retries': 19, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 135, 'classified_keyword_fallback': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4}, 'route_counts': {'support_queue': 69, 'premium_desk': 44, 'security_team': 34, 'human_review': 17}}


,status,route,tickets
0,classified_json,premium_desk,43
1,classified_json,security_team,31
2,classified_json,support_queue,61
3,classified_keyword_fallback,premium_desk,1
4,classified_keyword_fallback,security_team,3
5,classified_keyword_fallback,support_queue,8
6,flagged_invalid_output,human_review,9
7,flagged_long,human_review,4
8,skipped_empty,human_review,4


,ticket_id,customer_id,true_category,prompt_mode,status,predicted_category,priority,route,tokens_used,retry_count,pii_items_redacted
0,T0024,C0486,fraud,few_shot,classified_json,fraud,HIGH,security_team,79,0,2
1,T0073,C0080,billing,few_shot,classified_json,billing,HIGH,premium_desk,75,0,2
2,T0095,C1087,technical,few_shot,classified_json,technical,NORMAL,support_queue,77,0,1
3,T0159,C1172,none,few_shot,skipped_empty,,NORMAL,human_review,0,0,0
4,T0080,C0910,fraud,few_shot,classified_json,fraud,HIGH,security_team,79,0,2


## 2. Grounded policy RAG

Five local policy documents are split into overlapping chunks and indexed with TF-IDF. Answers quote a retrieved policy sentence and cite its file; unsupported questions return an explicit fallback.

In [3]:
rag_result = run_policy_rag()
print(rag_result["summary"])
display(rag_result["evaluation"])
example = rag_result["rag"].answer("Who approves a refund above INR 1000?")
print("Example answer:", example["answer"])
print("Source:", example["sources"])

{'questions': 14, 'supported_questions': 12, 'unsupported_questions': 2, 'source_accuracy': 1.0, 'fact_accuracy': 1.0, 'fallback_accuracy': 1.0, 'chunks': 12, 'documents': 5}


,question,expected_doc,answer,sources,source_match,fact_match,fallback,top_score
0,What delivery time makes a completed order eli...,refund_policy.txt,If a completed order is delivered in more than...,refund_policy.txt,True,True,False,0.445085
1,Who must approve a refund above INR 1000?,refund_policy.txt,A refund above INR 1000 must be approved by a ...,refund_policy.txt,True,True,False,0.489277
2,How many days do I have to raise a refund requ...,refund_policy.txt,A refund request must be raised within 7 days ...,refund_policy.txt,True,True,False,0.314981
3,What is the delivery fee on small orders?,delivery_policy.txt,A delivery fee of INR 25 is charged on orders ...,delivery_policy.txt,True,True,False,0.306689
4,On orders of what value is delivery free?,delivery_policy.txt,A delivery fee of INR 25 is charged on orders ...,delivery_policy.txt,True,True,False,0.257192
5,Why does the electronics delivery need a one t...,delivery_policy.txt,Electronics items are delivered only after the...,delivery_policy.txt,True,True,False,0.423628
6,Is cash on delivery available for a large order?,payment_policy.txt,Cash on delivery is available only for orders ...,payment_policy.txt,True,True,False,0.409140
7,How soon is money returned when a payment fail...,payment_policy.txt,"If a payment fails but money is deducted, the ...",payment_policy.txt,True,True,False,0.298782
8,Can wallet balance be refunded as cash?,payment_policy.txt,Wallet balance can be used for any order and i...,payment_policy.txt,True,True,False,0.566522
9,When are delivery addresses deleted from activ...,privacy_policy.txt,Delivery addresses are deleted from active sys...,privacy_policy.txt,True,True,False,0.415970


Example answer: A refund above INR 1000 must be approved by a human support lead before money is returned.
Source: ['refund_policy.txt']


## 3. Bounded refund agent

The agent retrieves an order from SQLite, searches policy, calculates eligibility, and either issues a simulated refund or escalates to a human. It enforces step/token limits and uses an idempotent append-only ledger.

In [4]:
agent_evaluation, agent_summary = evaluate_refund_agent()
print(agent_summary)
display(agent_evaluation)
assert agent_evaluation["matches"].all()
print("ALL SUPPORT COPILOT CHECKPOINTS PASSED")

{'scenarios': 9, 'passed': 9, 'accuracy': 1.0, 'max_steps': 6, 'automatic_refund_limit_inr': 1000, 'late_delivery_threshold_minutes': 45}


,scenario,order_id,expected_status,actual_status,matches,tool_sequence,final_answer
0,automatic_refund,ORD-100048,REFUNDED,REFUNDED,True,get_order>search_policy>calculate_refund>issue...,Refund of INR 128.25 was issued to the origina...
1,human_approval,ORD-100145,ESCALATED,ESCALATED,True,get_order>search_policy>calculate_refund>escal...,Refund of INR 1999.00 requires human approval.
2,on_time_denial,ORD-100001,DENIED,DENIED,True,get_order>search_policy>calculate_refund,Refund denied: delivery_not_over_45_minutes.
3,cancelled_denial,ORD-100020,DENIED,DENIED,True,get_order>search_policy>calculate_refund,Refund denied: only_completed_orders_are_eligi...
4,expired_request,ORD-100048,DENIED,DENIED,True,get_order>search_policy>calculate_refund,Refund denied: request_outside_7_day_window.
5,not_found,ORD-999999,NOT_FOUND_OR_FORBIDDEN,NOT_FOUND_OR_FORBIDDEN,True,get_order,The order was not found for the authenticated ...
6,wrong_customer,ORD-100048,NOT_FOUND_OR_FORBIDDEN,NOT_FOUND_OR_FORBIDDEN,True,get_order,The order was not found for the authenticated ...
7,missing_order_id,,NEEDS_INFO,NEEDS_INFO,True,,Please provide an order ID in the form ORD-123...
8,missing_identity,ORD-100048,AUTH_REQUIRED,AUTH_REQUIRED,True,,Authenticated customer identity is required.


ALL SUPPORT COPILOT CHECKPOINTS PASSED
